# EURS AI Studio: train on the official EURS data (Google Colab GPU)

Runtime → Change runtime type → **GPU** (an A100 or L4 is ideal; a T4 works with `--batch 4`).

This notebook runs the full protocol: dataset indexing → leakage-aware train/val/test split → EURS-Lite + EURS-Net training → held-out test evaluation → paper tables/figures.

In [ ]:
!nvidia-smi
!git clone https://github.com/nvaltryek-001/EURS-ai-studio.git
%cd EURS-ai-studio
!pip -q install -r ml/requirements-torch.txt

## 1. Get the data
Open the [official EURS folder](https://drive.google.com/drive/folders/10nWoxDEdWjKoIzYQp3--KSnWca3Vn4C5) → right-click → **Organize → Add shortcut** to *My Drive*. Then mount Drive and set `EURS_DIR` to that shortcut. The training archive is ~22 GB, so copying to local disk takes a while; Colab's disk is large enough.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EURS_DIR = '/content/drive/MyDrive/EURS'  # <- edit to your shortcut name
!ls -la "$EURS_DIR"

In [ ]:
!mkdir -p data/eurs_train
!cp "$EURS_DIR/eurs_train.zip" /content/eurs_train.zip
!unzip -q /content/eurs_train.zip -d data/eurs_train && rm /content/eurs_train.zip
!find data/eurs_train -maxdepth 3 -type d | head -50
!find data/eurs_train -type f | head -20

## 2. Index + split
Check the printed counts: `with_clean` and `with_mask` should be non-zero, and there should be no warning about unknown mask pixels. If auto-detection picks the wrong folders, copy `ml/configs/layout.example.json`, edit it, and pass `--layout`.

In [ ]:
!python ml/prepare_dataset.py --root data/eurs_train --out data/splits/eurs

## 3. Train, evaluate, report
`NET_ARGS` are passed to `ml/train_net.py`. Ablations for the paper are in the next cell.

In [ ]:
%env NET_ARGS=--encoder resnet18 --epochs 60 --batch 8 --size 384
!bash ml/run_pipeline.sh data/eurs_train eurs

In [ ]:
# Optional ablations (each ~same cost as the main run)
!python ml/train_net.py --manifest data/splits/eurs/manifest.csv --out runs/abl_nofusion --no-fusion --epochs 60
!python ml/export_onnx.py --ckpt runs/abl_nofusion/best.pt --out runs/abl_nofusion/eurs_net_nofusion.onnx --name eurs_net_nofusion
!python ml/evaluate.py --manifest data/splits/eurs/manifest.csv --model runs/abl_nofusion/eurs_net_nofusion.onnx --out results/eurs_ablation/no_fusion
!python ml/train_net.py --manifest data/splits/eurs/manifest.csv --out runs/abl_mbv3 --encoder mobilenet_v3_large --epochs 60
!python ml/export_onnx.py --ckpt runs/abl_mbv3/best.pt --out runs/abl_mbv3/eurs_net_mbv3.onnx --name eurs_net_mbv3
!python ml/evaluate.py --manifest data/splits/eurs/manifest.csv --model runs/abl_mbv3/eurs_net_mbv3.onnx --out results/eurs_ablation/mobilenet_v3

## 4. Save everything back to Drive
Commit `results/eurs`, `paper/generated`, `paper/figures` and `frontend/public/results.json` to the repo. To serve the model from the API, copy `runs/eurs_net/eurs_net.onnx` + `.json` into `backend/models/`.

In [ ]:
!mkdir -p "$EURS_DIR/outputs"
!zip -qr /content/eurs_outputs.zip results paper/generated paper/figures frontend/public/results.json runs/*/train_log.json runs/*/*.onnx runs/*/*.json runs/*/*.npz data/splits
!cp /content/eurs_outputs.zip "$EURS_DIR/outputs/"
print('saved to', EURS_DIR + '/outputs/eurs_outputs.zip')

## 5. Challenge predictions on the dev set (no ground truth; never use it for training)

In [ ]:
!cp "$EURS_DIR/eurs_dev_degraded.zip" /content/ && mkdir -p data/eurs_dev && unzip -q /content/eurs_dev_degraded.zip -d data/eurs_dev
!python ml/predict.py --model runs/eurs_net/eurs_net.onnx --input data/eurs_dev --out submissions/dev
# Check the challenge page for the exact submission format before uploading.